# S44_05 — BERT, GPT, and T5 Architecture Families

Three architectural families dominate — the choice determines what a model is good at.

## Encoder-only: BERT family

**Architecture:** Stack of bidirectional attention blocks. Every token sees every other token.  
**Pre-training:** Masked Language Modelling (MLM) — randomly mask 15% of tokens, predict them.  
**Best for:** tasks that require *understanding* a full input — classification, NER, extractive Q&A, embeddings.

| Model | Params | Notes |
|-------|--------|-------|
| BERT-base | 110M | Original (2018) |
| RoBERTa | 125M | Better training, still widely used |
| DeBERTa-v3 | 183M | State-of-the-art for classification tasks |
| ModernBERT | 149M | 2024, trained on 2T tokens, long context (8192) |

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# BERT for sentence embeddings (mean pooling)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

sentences = ['The quick brown fox.', 'A fast auburn canine.']
inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# Mean pool over sequence dimension (ignoring padding)
mask = inputs['attention_mask'].unsqueeze(-1).float()
embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
print(f'Embedding shape: {embeddings.shape}')  # (2, 768)

sim = torch.cosine_similarity(embeddings[0], embeddings[1], dim=0)
print(f'Cosine similarity: {sim:.3f}')  # high — semantically similar

## Decoder-only: GPT family

**Architecture:** Stack of causal (masked) attention blocks. Each token only sees past tokens.  
**Pre-training:** Causal Language Modelling (CLM) — predict the next token.  
**Best for:** text generation, reasoning, chat, code. The dominant LLM architecture.

| Model | Params | Notes |
|-------|--------|-------|
| GPT-2 | 117M–1.5B | Classic; still useful for learning |
| Llama 3.2 | 1B–405B | Meta's open-weight; best open performance |
| Mistral 7B | 7B | Efficient; strong performance per parameter |
| Qwen2.5 | 0.5B–72B | Multilingual; strong on code/math |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# GPT-2 generation
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

prompt = 'The transformer architecture works by'
inputs = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

## Encoder-Decoder: T5 family

**Architecture:** Bidirectional encoder + causal decoder with cross-attention.  
**Pre-training:** Span corruption — mask spans of text and train to reconstruct them.  
**Best for:** tasks framed as input→output transformations: translation, summarisation, Q&A, data-to-text.

| Model | Params | Notes |
|-------|--------|-------|
| T5-base | 220M | Original (2019) |
| FLAN-T5 | 80M–11B | Instruction-tuned; excellent for seq2seq tasks |
| BART | 140M | Better at summarisation than T5 |

## Decision guide

```
Task requires understanding/classification/embeddings?
  → Encoder-only: DeBERTa-v3, ModernBERT, sentence-transformers

Task requires generation/chat/reasoning?
  → Decoder-only: Llama 3, Mistral, Qwen2.5, GPT-4o, Claude

Task is explicitly sequence-to-sequence (translate, summarise)?
  → Encoder-Decoder: FLAN-T5, BART
  (though modern large decoder-only models can do this too)
```
